###Crear la tabla results_movie en la capa Gold

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold.results_movie(
              year_release_date INT,
              country_name STRING,
              company_name STRING,
              budget FLOAT,
              revenue FLOAT,
              movie_id INT,
              country_id INT,
              company_id INT,
              created_date DATE,
              updated_date DATE
          )
          USING DELTA
          """)

DataFrame[]

In [0]:
spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW v_results_movie
            AS
            SELECT M.year_release_date, C.country_name, PCO.company_name, M.budget, M.revenue, M.movie_id, C.country_id, PCO.company_id
            FROM movie_silver.movies M
            INNER JOIN movie_silver.productions_countries PC ON M.movie_id = PC.movie_id
            INNER JOIN movie_silver.countries C ON C.country_id = PC.country_id
            INNER JOIN movie_silver.movies_companies MC ON M.movie_id = MC.movie_id
            INNER JOIN movie_silver.productions_companies PCO ON MC.company_id = PCO.company_id
            WHERE M.file_date = '{v_file_date}'
""")

DataFrame[]

In [0]:
spark.sql("""
            MERGE INTO movie_gold.results_movie tgt
            USING v_results_movie src
            ON (tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id)
            WHEN MATCHED THEN
            UPDATE SET
                tgt.year_release_date = src.year_release_date,
                tgt.country_name = src.country_name,
                tgt.company_name = src.company_name,
                tgt.budget = src.budget,
                tgt.revenue = src.revenue,
                tgt.updated_date = current_timestamp
            WHEN NOT MATCHED
            THEN INSERT ( year_release_date, country_name, company_name, budget, revenue,
            movie_id, country_id, company_id, created_date)
            VALUES ( year_release_date, country_name, company_name, budget, revenue,
            movie_id, country_id, company_id, current_timestamp)
""")

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
586,586,0,0


In [0]:
%sql
SELECT COUNT(*) FROM v_results_movie;

count(1)
586


In [0]:
%sql
SELECT COUNT(*) FROM movie_gold.results_movie;

count(1)
586


In [0]:
%sql
SELECT * FROM movie_gold.results_movie;

year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id,created_date,updated_date
null,United States of America,Carolina Bank,3.15E7,1979.0,28,214,504,2026-09-28,2026-09-28
1985,United Kingdom,Pandora Filmproduktion,1.5E7,1.875E7,68,162,254,2026-09-28,2026-09-28
2000,United Kingdom,Zentropa Entertainments,1.03E8,4.57640416E8,98,162,76,2026-09-28,2026-09-28
2000,United States of America,Zentropa Entertainments,1.03E8,4.57640416E8,98,214,76,2026-09-28,2026-09-28
1998,United Kingdom,Samuel Goldwyn,1350000.0,3897569.0,100,162,261,2026-09-28,2026-09-28
2001,United States of America,Davis-Films,8.5E7,4.50717152E8,161,214,342,2026-09-28,2026-09-28
2006,Germany,DOS,2.7E7,5.630888E7,186,151,24137,2026-09-28,2026-09-28
2006,United States of America,DOS,2.7E7,5.630888E7,186,214,24137,2026-09-28,2026-09-28
2001,France,O2 Filmes,1.0E7,1.73921952E8,194,161,345,2026-09-28,2026-09-28
2001,Germany,O2 Filmes,1.0E7,1.73921952E8,194,151,345,2026-09-28,2026-09-28


In [0]:
%sql
SELECT * FROM v_results_movie;

year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id
null,United States of America,Carolina Bank,3.15E7,1979.0,28,214,504
1985,United Kingdom,Pandora Filmproduktion,1.5E7,1.875E7,68,162,254
2000,United Kingdom,Zentropa Entertainments,1.03E8,4.57640427E8,98,162,76
2000,United States of America,Zentropa Entertainments,1.03E8,4.57640427E8,98,214,76
1998,United Kingdom,Samuel Goldwyn,1350000.0,3897569.0,100,162,261
2001,United States of America,Davis-Films,8.5E7,4.5071715E8,161,214,342
2006,Germany,DOS,2.7E7,5.6308881E7,186,151,24137
2006,United States of America,DOS,2.7E7,5.6308881E7,186,214,24137
2001,France,O2 Filmes,1.0E7,1.73921954E8,194,161,345
2001,Germany,O2 Filmes,1.0E7,1.73921954E8,194,151,345
